# Multi-agent listingIQ — Credits agent + Listings agent + Orchestrator

Three agents, meant to run in SageMaker JupyterLab, calling the real AgentCore Gateway built in `infra/` (see `infra/README.md` for how that Gateway was provisioned):

```
Agent 1 (credits specialist) --tools--> credits-api MCP tools --> Gateway --> Lambda --> atlas.propertyfinder.com
Agent 2 (listings specialist) --tools--> listings-api MCP tools --> Gateway --> Lambda --> atlas.propertyfinder.com
Agent 3 (orchestrator) --tools--> [Agent 1, Agent 2] --> combines results --> answer
```

This is a **side-by-side track** with the CSV-based demo agent (`agent.py`) — it does not replace it. Nothing here touches `agent.py`, `tools.py`, or `demo_data.py`.

**Before running:** the Gateway's Secrets Manager secret (`listingiq/enterprise-api-credentials`) still holds placeholder values — see `infra/README.md`. Until a real enterprise-api `apiKey`/`apiSecret` is set, `tools/call` invocations here will fail at the token-mint step even though the Gateway and tool discovery work correctly.

Auth: the Gateway's inbound auth is `AWS_IAM` (SigV4) — this notebook needs AWS credentials for the hackathon account (`aws sso login --profile pf-hackathon`).

In [ ]:
%pip install -q strands-agents mcp-proxy-for-aws boto3

In [ ]:
from mcp_proxy_for_aws.client import aws_iam_streamablehttp_client
from strands import Agent, tool
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient

# -- Config: matches infra/deploy.py's provisioned Gateway --------------------
GATEWAY_URL = "https://listingiq-gateway-pjolmk6vdb.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp"
AWS_REGION = "us-east-1"
AWS_PROFILE = "pf-hackathon"  # omit / set to None if running with default credentials in SageMaker

model = BedrockModel(model_id="qwen.qwen3-coder-next", region_name=AWS_REGION)

## Agent 1 — Credits specialist

Connects to the same Gateway but `tool_filters` restricts it to only the `credits-api` tools, so this agent can't accidentally touch listings.

In [ ]:
credits_mcp_client = MCPClient(
    lambda: aws_iam_streamablehttp_client(
        endpoint=GATEWAY_URL,
        aws_region=AWS_REGION,
        aws_service="bedrock-agentcore",
        aws_profile=AWS_PROFILE,
    ),
    tool_filters={"allowed": ["credits-api___get_credit_balance", "credits-api___get_credit_transactions"]},
)


@tool
def credits_specialist(query: str) -> str:
    """
    Answer questions about credit balance and credit spend history.
    Use this for anything about credits, spending, or transactions.

    Args:
        query: The credits-related question.

    Returns:
        The credits specialist agent's answer.
    """
    try:
        agent = Agent(
            model=model,
            system_prompt=(
                "You are a credits specialist for a real estate platform. "
                "Use get_credit_balance and get_credit_transactions to answer questions about "
                "credit balance and spend. Be concise and cite specific numbers."
            ),
            tools=[credits_mcp_client],
        )
        return str(agent(query))
    except Exception as e:
        return f"Error in credits_specialist: {e}"

## Agent 2 — Listings specialist

Same Gateway, filtered to `listings-api` tools only.

In [ ]:
listings_mcp_client = MCPClient(
    lambda: aws_iam_streamablehttp_client(
        endpoint=GATEWAY_URL,
        aws_region=AWS_REGION,
        aws_service="bedrock-agentcore",
        aws_profile=AWS_PROFILE,
    ),
    tool_filters={"allowed": ["listings-api___search_listings", "listings-api___get_listing"]},
)


@tool
def listings_specialist(query: str) -> str:
    """
    Answer questions about listings: search, filters, and individual listing details.
    Use this for anything about listings, not credits.

    Args:
        query: The listings-related question.

    Returns:
        The listings specialist agent's answer.
    """
    try:
        agent = Agent(
            model=model,
            system_prompt=(
                "You are a listings specialist for a real estate platform. "
                "Use search_listings and get_listing to answer questions about listings. "
                "Be concise and cite specific listing IDs/details."
            ),
            tools=[listings_mcp_client],
        )
        return str(agent(query))
    except Exception as e:
        return f"Error in listings_specialist: {e}"

## Agent 3 — Orchestrator

Routes to Agent 1 and/or Agent 2 depending on the question, and combines their answers when a question spans both (e.g. "credits spent on my listings in Business Bay").

In [ ]:
orchestrator = Agent(
    model=model,
    system_prompt=(
        "You are an assistant that routes real estate questions to specialized agents:\n"
        "- For credit balance/spend questions -> use credits_specialist\n"
        "- For listing search/detail questions -> use listings_specialist\n"
        "- For questions spanning both -> call both and combine the results into one answer\n"
        "Always explain which specialist(s) you used."
    ),
    tools=[credits_specialist, listings_specialist],
)

In [ ]:
# Smoke test. Expect an auth error surfaced from the tool call until the
# Secrets Manager secret has real enterprise-api credentials (see infra/README.md).
response = orchestrator("What is our current credit balance?")
print(response)

In [ ]:
# A question that should route to both specialists
response = orchestrator("Search for live listings in Dubai and tell me our credit balance")
print(response)